# 09 — Performance statistics and report

The slice-wide IC backtest's headline statistics — mean IC, IC standard deviation, IC information ratio, hit rate and mean decile spread — plus the rendered `build_backtest_report` text.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## The statistics this strategy has — and the ones it does not

**Has:** Information Coefficient (Spearman rank correlation between score and realized forward return), its standard deviation and information ratio, hit rate (fraction of cycles with positive IC), and the decile spread.

**Does not have, at all:** total return, CAGR, Sharpe ratio, drawdown, equity curve, turnover. Those are P&L-portfolio concepts and this strategy never holds a position.

**Also does not have: AUC.** ROC AUC measured a binary classifier's ability to separate two classes. There are no classes under LambdaRank, so AUC is not merely unreported here — it was deleted from the codebase along with the classifier.

In [ ]:
if AVAILABLE:
    result = rd.slice_bundle().ic_result
    print("cycles evaluated ", result.cycle_count)
    print("cycles measured  ", result.measured_cycle_count,
          f"(>= {result.min_scored_count} scored candidates and a realized next cycle)")
else:
    result = None
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Headline statistics

In [ ]:
def fmt(value, precision=4):
    return "—" if value is None else f"{value:.{precision}f}"


if AVAILABLE:
    print(f"mean IC              {fmt(result.mean_ic)}")
    print(f"IC std dev           {fmt(result.ic_std)}")
    print(f"IC information ratio {fmt(result.ic_information_ratio)}")
    print(f"hit rate (IC > 0)    {'—' if result.hit_rate is None else f'{result.hit_rate:.1%}'}")
    print(f"mean decile spread   {fmt(result.mean_decile_spread)}")
    print()
    print("IC information ratio is mean_ic / ic_std. It is None — never a fabricated")
    print("infinity — when the standard deviation is zero or no cycle was measured.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Per-cycle detail

The aggregate is a mean over very few quarters, so the per-cycle spread matters more than the mean itself.

In [ ]:
if AVAILABLE:
    print(f"{'cycle':<12} {'training':<32} {'ranked':>8} {'scored':>8} "
          f"{'IC':>9} {'decile spread':>14}")
    for cycle_result in result.cycle_results:
        ic = "—" if cycle_result.ic is None else f"{cycle_result.ic:9.4f}"
        spread = "—" if cycle_result.decile_spread is None else f"{cycle_result.decile_spread:14.4f}"
        print(f"{str(cycle_result.cycle.quarter_start):<12} "
              f"{cycle_result.training_state.value:<32} "
              f"{cycle_result.ranked_count:>8,} {cycle_result.scored_for_ic_count:>8,} "
              f"{ic:>9} {spread:>14}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Reading the decile spread

Top-decile mean forward return minus bottom-decile mean forward return, after sorting by score. Purely descriptive: it never implies capital is deployed long the top decile and short the bottom one, and it is `None` when there are fewer than 10 pairs (deciles would be degenerate).

In [ ]:
if AVAILABLE:
    spreads = [
        (c.cycle.quarter_start, c.decile_spread)
        for c in result.cycle_results
        if c.decile_spread is not None
    ]
    for quarter_start, spread in spreads:
        bar = "#" * max(0, round(abs(spread) * 200))
        sign = "+" if spread >= 0 else "-"
        print(f"  {quarter_start}  {spread:>8.4f}  {sign}{bar}")
    print()
    positive = sum(1 for _, spread in spreads if spread > 0)
    print(f"{positive} of {len(spreads)} measured cycles had a positive decile spread.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## The rendered report

`build_backtest_report(...).to_text()` — the same renderer the CLI uses.

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.reporting.report_builder import (
        build_backtest_report,
    )

    print(build_backtest_report(result).to_text())
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Serialized result

`to_dict()` is the structured form — the authoritative record, of which the text above is only a rendering.

In [ ]:
if AVAILABLE:
    payload = result.to_dict()
    for key in (
        "strategy_id", "strategy_version", "config_identity", "run_identity",
        "cycle_count", "min_scored_count", "measured_cycle_count",
        "mean_ic", "ic_std", "ic_information_ratio", "hit_rate", "mean_decile_spread",
    ):
        print(f"  {key:<22} {payload[key]}")
    print()
    print("keys present:", sorted(payload))
    for forbidden in ("auc", "roc_auc", "sharpe", "total_return", "max_drawdown", "equity_curve"):
        assert forbidden not in payload, forbidden
    print()
    print("OK: no AUC, Sharpe, total-return, drawdown or equity-curve field exists.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Findings

- Over the measured cycles of this slice the ranking shows a positive mean IC with a positive decile spread — the ordering carried real forward information on this window.
- The IC information ratio is computed from a handful of quarters. It is a descriptive ratio over that sample, not a statistically established result.
- The serialized result is asserted to contain no AUC, Sharpe, return, drawdown or equity-curve field — those concepts do not exist in this strategy.

## Limitations

- **The sample is far too small to conclude anything.** A mean over a handful of quarterly ICs has a very wide confidence interval; no significance test is performed and none would pass at this sample size.
- The measured cycles are recent and the universe is survivorship-biased. Both bias these figures upward relative to what was achievable in real time.
- Hyperparameters were carried over from the previous classifier and are untuned; there is no held-out validation and no multiple-comparison adjustment.
- The decile spread is a gross, cost-free, price-only difference of means. It is not a return: there are no transaction costs, no shorting, no borrow, and no capital in this strategy at all.